> Projeto Desenvolve <br>
Programação Intermediária com Python <br>
Profa. Camila Laranjeira (mila@projetodesenvolve.com.br) <br>

# 3.11 - Data Model

## Exercícios

#### Q1. `dataclass`
Exercício adaptado de [codechalleng.es/bites/154/](https://codechalleng.es/bites/154/) e [codechalleng.es/bites/320/](https://codechalleng.es/bites/320/).

Neste desafio, você deve escrever uma `dataclass` chamada `Bite` que gerencia 3 atributos: `number`, `title` e `level`. Seus tipos são:
* `number` - `int`, 
* `title` - `str`, 
* `level` -  classe `Enum` chamada `BiteLevel` com os atributos `Beginner`, `Intermediate`, `Advanced`. 

Exemplo de dado: `{'number': 154, 'title': 'Escreva uma dataclass', 'level': BiteLevel.Intermediate}`

As características dessa classe são:
* O atributo`level` tem um valor padrão `BiteLevel.Beginner`
* Uma coleção de objetos `Bite` tem que ser ordenável somente pelo atributo `number`
* Implemente o método especial `__str__` para imprimir o Bite na forma `f'{number} - {title} ({level})'`

Teste sua classe executando o seguinte código:
```python
bites = []
bites.append(Bite(154, 'Escreva uma dataclass', 'Intermediate'))
bites.append(Bite(1, 'Some n valores'))
bites.append(Bite(37, 'Reescreva um loop com recursão', 'Intermediate'))

for b in bites.sort(): print(b)
# Ordem esperada na saída:
# 1 - Some n valores (Beginner)
# 37 - Reescreva um loop com recursão (Intermediate)
# 154 - Escreva uma dataclass (Intermediate)
```

In [ ]:
#### Escreva sua resposta aqui
from dataclasses import dataclass, field
from enum import Enum

class BiteLevel(Enum):
    Beginner = "Beginner"
    Intermediate = "Intermediate"
    Advanced = "Advanced"

    def __str__(self):
        return self.value

@dataclass(order=True)
class Bite:
    # A ordem de definição importa para a comparação automática
    number: int
    title: str = field(compare=False)
    level: BiteLevel = field(default=BiteLevel.Beginner, compare=False)

    def __post_init__(self):
        # Garante que se o level for passado como string, vire o Enum correspondente
        if isinstance(self.level, str):
            self.level = BiteLevel[self.level]

    def __str__(self):
        return f'{self.number} - {self.title} ({self.level})'

# Teste da classe
bites = []
bites.append(Bite(154, 'Escreva uma dataclass', 'Intermediate'))
bites.append(Bite(1, 'Some n valores'))
bites.append(Bite(37, 'Loop com recursão', 'Intermediate'))

# Ordena a lista (in-place) e imprime
bites.sort()
for b in bites:
    print(b)

#### Q2. `Pydantic`
> Adaptada desse [tutorial de Pydantic](https://github.com/adonath/scipy-2023-pydantic-tutorial/tree/main) criado por [Axel Donath](https://github.com/adonath) e [Nick Langellier](https://github.com/nlangellier).

Observe a seguinte lista de observações da previsão do tempo em Murmansk, Russia.
```python
data_samples = [
    {
        "date": "2023-05-20",
        "temperature": 62.2,
        "isCelsius": False,
        "airQualityIndex": "24",
        "sunriseTime": "01:26",
        "sunsetTime": "00:00",
    },
    {
        "date": "2023-05-21",
        "temperature": "64.4",
        "isCelsius": "not true",
        "airQualityIndex": 23,
        "sunriseTime": "01:10",
        "sunsetTime": "00:16",
    },
    {
        "date": "2023-05-22",
        "temperature": 14.4,
        "airQualityIndex": 21,
    },
]
```

Escreva um script que calcule e imprima a temperatura média (em Celsius) em Murmansk para as datas fornecidas. Em seu script, você deve incluir um modelo Pydantic que registre com sucesso todos os elementos dados. Note que:

* Algumas amostras estão faltando dados. Você deve decidir quando o atributo pode ter um valor padrão ou quando definí-lo como opcional (`typing.Optional`). 
* Você precisará implementar pelo menos um validador de campo para transformar atributos. Dica: teste primeiro quais vão falhar :)



In [ ]:
#### Escreva sua resposta aqui
from pydantic import BaseModel, field_validator, Field
from typing import Optional, List
from datetime import date, time

class WeatherSample(BaseModel):
    date: date
    temperature: float
    isCelsius: bool = True  # Valor padrão
    airQualityIndex: int
    sunriseTime: Optional[time] = None
    sunsetTime: Optional[time] = None

    @field_validator('isCelsius', mode='before')
    @classmethod
    def validate_bool(cls, v):
        if isinstance(v, str):
            if v.lower() == 'not true': return False
            if v.lower() == 'true': return True
        return v

data_samples = [
    {"date": "2023-05-20", "temperature": 62.2, "isCelsius": False, "airQualityIndex": "24", "sunriseTime": "01:26", "sunsetTime": "00:00"},
    {"date": "2023-05-21", "temperature": "64.4", "isCelsius": "not true", "airQualityIndex": 23, "sunriseTime": "01:10", "sunsetTime": "00:16"},
    {"date": "2023-05-22", "temperature": 14.4, "airQualityIndex": 21},
]

# Processamento
parsed_samples = [WeatherSample(**s) for s in data_samples]
temps_celsius = []

for s in parsed_samples:
    temp = s.temperature
    if not s.isCelsius:
        temp = (temp - 32) * 5 / 9
    temps_celsius.append(temp)

avg_temp = sum(temps_celsius) / len(temps_celsius)
print(f"Temperatura média em Murmansk: {avg_temp:.2f}°C")

#### Q3
> Adaptada desse [tutorial de Pydantic](https://github.com/adonath/scipy-2023-pydantic-tutorial/tree/main) criado por [Axel Donath](https://github.com/adonath) e [Nick Langellier](https://github.com/nlangellier).

Na célula a seguir, coletamos dados reais de uma das principais APIs de previsão do tempo, [open-meteo](https://open-meteo.com/en/docs). Não se preocupe em entender esse código, o mais importante é entender o resultado que ele retorna, ilustrado a seguir para uma coleta da temperatura dos últimos 15 dias em Itabira -MG. Caso deseje alterar a cidade de coleta, basta alimentar a latitude e longitude desejada, como nas opções a seguir.
* Itabira: `'latitude': -19.656655787605846, 'longitude': -43.228922960534476`
* Bom Despacho: `'latitude': -19.726308457732443, 'longitude': -45.27462803349767`

```python
{
  "latitude": -19.5,
  "longitude": -43.375,
  "generationtime_ms": 0.01800060272216797,
  "utc_offset_seconds": 0,
  "timezone": "GMT",
  "timezone_abbreviation": "GMT",
  "elevation": 2.0,
  "hourly_units": {
    "time": "iso8601",
    "temperature_2m": "\u00b0C"
  },
  "hourly": {
    "time": [
      "2024-07-19T00:00",
      "2024-07-19T01:00",
      "2024-07-19T02:00",
      ...
    ],
    "temperature_2m": [
      21.9,
      20.9,
      20.0,
      ... 
    ]
  }
}
```

Você deve escrever um modelo Pydantic `OpenMeteo` que receba diretamente a resposta dessa API, através do comando:
```python
dados = OpenMeteo(**response)
``` 

Para comportar a estrutura hierárquica desse dicionário (é um dicionário com alguns dicionários internos), você deve criar uma classe Pydantic para cada dicionário interno (`HourlyUnits` e `Hourly`), com seus respectivos atributos. Essas classes serão atributos da classe principal `OpenMeteo`, que terá também os outros atributos da resposta (`latitude`, `longitude`, etc.).



In [ ]:
import requests, json

url = 'https://api.open-meteo.com/v1/forecast'
lat, long = -19.656655787605846, -43.228922960534476
params = {'latitude': lat, 'longitude': long, 'elevation': 2,
          'hourly': 'temperature_2m', 'forecast_days': 15}
response = requests.get(url, params=params).json()
print(json.dumps(response, indent=2))

In [ ]:
#### Escreva aqui seus modelos Pydantic
from pydantic import BaseModel
from typing import List
import matplotlib.pyplot as plt

class HourlyUnits(BaseModel):
    time: str
    temperature_2m: str

class Hourly(BaseModel):
    time: List[str]
    temperature_2m: List[float]

class OpenMeteo(BaseModel):
    latitude: float
    longitude: float
    generationtime_ms: float
    utc_offset_seconds: int
    timezone: str
    timezone_abbreviation: str
    elevation: float
    hourly_units: HourlyUnits
    hourly: Hourly

# Simulação da resposta da API (exemplo fornecido)
response = {
    "latitude": -19.5, "longitude": -43.375, "generationtime_ms": 0.018,
    "utc_offset_seconds": 0, "timezone": "GMT", "timezone_abbreviation": "GMT",
    "elevation": 2.0,
    "hourly_units": {"time": "iso8601", "temperature_2m": "°C"},
    "hourly": {
        "time": ["2024-07-19T00:00", "2024-07-19T01:00", "2024-07-19T02:00"],
        "temperature_2m": [21.9, 20.9, 20.0]
    }
}

# Carregamento no modelo
dados = OpenMeteo(**response)

# Visualização (Matplotlib)
plt.figure(figsize=(12, 5))
plt.plot(dados.hourly.time, dados.hourly.temperature_2m, marker='o', linestyle='-', color='b')
plt.title(f'Temperatura em Itabira/Bom Despacho (Lat: {dados.latitude})')
plt.xlabel('Timestamp')
plt.ylabel(f'Temperatura ({dados.hourly_units.temperature_2m})')
plt.xticks(rotation=45)
plt.grid(True)
plt.tight_layout()
plt.show()

#### Q4. 

Com os dados carregados na questão anterior plote um gráfico de linha, com a biblioteca de sua preferência, onde o eixo `x` são os timestamps (data e hora) e o eixo `y` é a temperatura medida.

In [ ]:
#### Escreva aqui a sua resposta
import matplotlib.pyplot as plt

# 1. Extração dos dados do objeto Pydantic 'dados'
eixo_x = dados.hourly.time
eixo_y = dados.hourly.temperature_2m
unidade = dados.hourly_units.temperature_2m

# 2. Configuração do gráfico
plt.figure(figsize=(12, 6))

# Plotagem da linha
plt.plot(eixo_x, eixo_y, marker='o', linestyle='-', color='#1f77b4', linewidth=2, markersize=4)

# 3. Customização de títulos e rótulos
plt.title(f'Variação de Temperatura - Itabira/Bom Despacho\n(Latitude: {dados.latitude}, Longitude: {dados.longitude})', fontsize=14)
plt.xlabel('Data e Hora (ISO8601)', fontsize=12)
plt.ylabel(f'Temperatura ({unidade})', fontsize=12)

# Rotacionar os labels do eixo X para melhor leitura
plt.xticks(rotation=45, ha='right')

# Adicionar grade para facilitar a visualização dos valores
plt.grid(True, linestyle='--', alpha=0.7)

# Ajustar o layout para não cortar os rótulos
plt.tight_layout()

# 4. Exibição
plt.show()